In [9]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.model_selection import GridSearchCV
from imblearn.ensemble import BalancedRandomForestClassifier
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier


from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix)

In [2]:
TH_500 = pd.read_csv(
    "../classification/TomsHardware/Relative_labeling/sigma=500/TomsHardware-Relative-Sigma-500.data",
    sep=",",
    header=None
)

TH_1000= pd.read_csv(
    "../classification/TomsHardware/Relative_labeling/sigma=1000/TomsHardware-Relative-Sigma-1000.data",
    sep=",",
    header=None
)
TH_1500= pd.read_csv(
    "../classification/TomsHardware/Relative_labeling/sigma=1500/TomsHardware-Relative-Sigma-1500.data",
    sep=",",
    header=None
)
groups = [
    "NCD", "BL", "NAD", "AI", "NAC", "ND",
    "CS", "AT", "NA", "ADL", "AS_NA", "AS_NAC"
]

columns = []
for group in groups:
    for t in range(8):
        columns.append(f"{group}_{t}")

columns.append("label")  

TH_500.columns = columns
TH_1000.columns = columns
TH_1500.columns = columns

prefixes = {col.split("_")[0] for col in TH_500.columns if "_" in col}
prefixes = {col.split("_")[0] for col in TH_1000.columns if "_" in col}
prefixes = {col.split("_")[0] for col in TH_1500.columns if "_" in col}

In [ ]:
1 Baseline Random Forest
2 Random Forest + Stratified K-Fold
3 Random Forest + Grid Search
4 Balanced Random Forest
5 Random Forest + SMOTE

### 500

### 1 Baseline Random Forest

In [8]:
X = TH_500.drop(columns=['label'])
y = TH_500['label']

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2, stratify=y,random_state=42)
rf = RandomForestClassifier(random_state=42).fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

#
accuracy_rf = accuracy_score(y_test, y_pred)
precision_rf = precision_score(y_test, y_pred)
recall_rf = recall_score(y_test, y_pred)
f1_rf = f1_score(y_test, y_pred)
roc_auc_rf = roc_auc_score(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

print('1.Baseline Random Forest')
print("Accuracy:", accuracy_rf)
print("Precision:", precision_rf)
print("Recall:", recall_rf)
print("F1-score:", f1_rf)
print("ROC-AUC:", roc_auc_rf)


print("Confusion Matrix:")
print(cm)

1.Baseline Random Forest
Accuracy: 0.9209361163820367
Precision: 0.8140845070422535
Recall: 0.8304597701149425
F1-score: 0.8221906116642959
ROC-AUC: 0.9683243840366922
Confusion Matrix:
[[1167   66]
 [  59  289]]


### 2.Random Forest with Stratified K-Fold

In [9]:
X = TH_500.drop(columns=['label'])
y = TH_500['label']

rf = RandomForestClassifier(random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}


cv_results = cross_validate( rf,X,y,cv=skf, scoring=scoring)

print('2.Random Forest with Stratified K-Fold')
print("Accuracy:", cv_results['test_accuracy'].mean())
print("Precision:", cv_results['test_precision'].mean())
print("Recall:", cv_results['test_recall'].mean())
print("F1-score:", cv_results['test_f1'].mean())
print("ROC-AUC:", cv_results['test_roc_auc'].mean())

2.Random Forest with Stratified K-Fold
Accuracy: 0.916382036685642
Precision: 0.8103210492609673
Recall: 0.8110216381780457
F1-score: 0.8103785135821081
ROC-AUC: 0.9636792211159024


### 3.Random Forest with Grid Search

In [13]:
X = TH_500.drop(columns=['label'])
y = TH_500['label']

rf = RandomForestClassifier(random_state=42)

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}


grid_search = GridSearchCV( estimator=rf,param_grid=param_grid,cv=5,scoring='f1',n_jobs=-1).fit(X_train, y_train)

best_rf = grid_search.best_estimator_

y_pred = best_rf.predict(X_test)
y_prob = best_rf.predict_proba(X_test)[:,1]

print('3.Random Forest with Grid Search')
print("Best parameters:", grid_search.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

3.Random Forest with Grid Search
Best parameters: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Accuracy: 0.920303605313093
Precision: 0.8083333333333333
Recall: 0.8362068965517241
F1-score: 0.8220338983050848
ROC-AUC: 0.9701177391839361


### 4.Balanced Random Forest

In [15]:
brf = BalancedRandomForestClassifier(n_estimators=200,random_state=42).fit(X_train, y_train)

y_pred = brf.predict(X_test)
y_prob = brf.predict_proba(X_test)[:,1]

accuracy_brf = accuracy_score(y_test, y_pred)
precision_brf = precision_score(y_test, y_pred)
recall_brf = recall_score(y_test, y_pred)
f1_brf = f1_score(y_test, y_pred)
roc_auc_brf = roc_auc_score(y_test, y_prob)

print("4. Balanced Random Forest")
print("Accuracy:", accuracy_brf)
print("Precision:", precision_brf)
print("Recall:", recall_brf)
print("F1-score:", f1_brf)
print("ROC-AUC:", roc_auc_brf)


cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

4. Balanced Random Forest
Accuracy: 0.9013282732447818
Precision: 0.711453744493392
Recall: 0.9281609195402298
F1-score: 0.8054862842892768
ROC-AUC: 0.969249610798818
Confusion Matrix:
[[1102  131]
 [  25  323]]


### 5. Random Forest with SMOTE and Grid Search

In [17]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix


pipeline = Pipeline([('smote', SMOTE(random_state=42)),('rf', RandomForestClassifier(random_state=42))])

param_grid = {
    'rf__n_estimators': [100, 200],
    'rf__max_depth': [None, 10, 20],
    'rf__min_samples_split': [2, 5],
    'rf__min_samples_leaf': [1, 2],
    'rf__max_features': ['sqrt', 'log2']
}

grid_search_smote = GridSearchCV(pipeline,param_grid,cv=5, scoring='f1',n_jobs=-1).fit(X_train, y_train)
best_smote_rf = grid_search_smote.best_estimator_

y_pred = best_smote_rf.predict(X_test)
y_prob = best_smote_rf.predict_proba(X_test)[:,1]

print("5. Random Forest with SMOTE and Grid Search")
print("Best parameters:", grid_search_smote.best_params_)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

cm5. Random Forest with SMOTE and Grid Search
Best parameters: {'rf__max_depth': 20, 'rf__max_features': 'sqrt', 'rf__min_samples_leaf': 1, 'rf__min_samples_split': 2, 'rf__n_estimators': 200}
Accuracy: 0.9127134724857685
Precision: 0.7573529411764706
Recall: 0.8879310344827587
F1-score: 0.8174603174603174
ROC-AUC: 0.9671264833925292
Confusion Matrix:
[[1134   99]
 [  39  309]] = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

5. Random Forest with SMOTE and Grid Search
Best parameters: {'rf__max_depth': 20, 'rf__max_features': 'sqrt', 'rf__min_samples_leaf': 1, 'rf__min_samples_split': 2, 'rf__n_estimators': 200}
Accuracy: 0.9127134724857685
Precision: 0.7573529411764706
Recall: 0.8879310344827587
F1-score: 0.8174603174603174
ROC-AUC: 0.9671264833925292
Confusion Matrix:
[[1134   99]
 [  39  309]]


Baseline Random Forest

Базова модель Random Forest продемонструвала досить високі результати. Значення accuracy склало 0.921, що свідчить про високу загальну точність класифікації. Показник precision дорівнює 0.814, а recall — 0.830, що означає, що модель достатньо добре ідентифікує події buzz та при цьому не створює надмірної кількості хибних спрацьовувань. Узагальнюючий показник F1-score становить 0.822, що підтверджує збалансованість між точністю та повнотою. Значення ROC-AUC дорівнює 0.968, що свідчить про дуже хорошу здатність моделі відрізняти класи.

Аналіз confusion matrix показує, що модель правильно класифікувала 1167 прикладів класу 0 та 289 прикладів класу 1, при цьому допустивши 66 хибно позитивних та 59 хибно негативних прогнозів. Це означає, що базова модель уже демонструє досить стабільну якість.

Random Forest з використанням Stratified K-Fold Cross-Validation

Застосування Stratified K-Fold cross-validation дозволило отримати більш надійну оцінку якості моделі. При цьому accuracy становить 0.916, що лише незначно відрізняється від результату базової моделі. Значення precision дорівнює 0.810, recall — 0.811, а F1-score — 0.810.

Порівняно з базовою моделлю, ці показники є трохи нижчими, що можна пояснити тим, що крос-валідація дає більш консервативну та стабільну оцінку продуктивності моделі на різних підмножинах даних. Значення ROC-AUC становить 0.964, що також підтверджує високу якість класифікації.

Random Forest з оптимізацією гіперпараметрів (Grid Search)

Для покращення якості моделі було проведено пошук оптимальних гіперпараметрів за допомогою Grid Search. Найкращою конфігурацією виявилася модель з параметрами:


Після оптимізації accuracy склало 0.920, що майже відповідає базовій моделі. Однак recall зріс до 0.836, що означає, що модель стала трохи краще виявляти події buzz. При цьому precision становить 0.808, а F1-score — 0.822, що майже повністю відповідає результату baseline моделі. Значення ROC-AUC дорівнює 0.970, що є найвищим показником серед усіх протестованих моделей і свідчить про відмінну здатність моделі відрізняти класи.

Balanced Random Forest

Для розв’язання проблеми незбалансованості класів було використано Balanced Random Forest. Цей підхід автоматично балансує вибірки під час побудови дерев.

Результати показали, що recall значно зріс до 0.928, що є найвищим показником серед усіх моделей. Це означає, що модель дуже ефективно знаходить приклади класу buzz. Однак при цьому precision знизився до 0.711, що свідчить про більшу кількість хибно позитивних прогнозів.

Загальна точність моделі (accuracy = 0.901) також трохи знизилася порівняно з попередніми моделями. F1-score становить 0.805, що трохи нижче за результати baseline Random Forest. Значення ROC-AUC = 0.969 все ще залишається дуже високим.

Confusion matrix показує, що модель правильно визначила 323 приклади класу buzz, але при цьому збільшила кількість хибно позитивних прогнозів до 131.

Random Forest з використанням SMOTE та Grid Search

Останнім етапом було використано SMOTE (Synthetic Minority Oversampling Technique) разом із оптимізацією гіперпараметрів. Найкраща модель мала параметри:

Результати показали, що recall становить 0.888, що значно вище за baseline модель, але трохи нижче, ніж у Balanced Random Forest. Precision дорівнює 0.757, що є компромісом між базовою моделлю та Balanced Random Forest. F1-score становить 0.817, що є досить високим значенням.

Загальна точність моделі (accuracy = 0.913) також залишається високою, а ROC-AUC дорівнює 0.967, що підтверджує хорошу здатність моделі розрізняти класи.

Порівняння моделей

Загалом результати показують, що базова модель Random Forest та модель з Grid Search демонструють найвищу загальну точність і F1-score. Водночас Balanced Random Forest забезпечує найвищий recall, що означає найкращу здатність знаходити події buzz, але ціною зниження precision.

Модель Random Forest із використанням SMOTE та Grid Search показує найбільш збалансований результат між precision та recall, що робить її хорошим компромісом між точністю та здатністю виявляти події buzz.

Таким чином, можна зробити висновок, що оптимізований Random Forest та Random Forest із SMOTE демонструють найкращий баланс між різними метриками, тоді як Balanced Random Forest є найбільш ефективним у виявленні рідкісних подій класу buzz.

Короткі висновки з таблиці

Найвищий ROC-AUC (0.970) показала модель Random Forest з Grid Search, що свідчить про найкращу здатність моделі відрізняти класи.

Найвищий Recall (0.928) має Balanced Random Forest, тобто ця модель найкраще знаходить події buzz.

Найвищі F1-score (0.822) продемонстрували Baseline Random Forest та Random Forest з Grid Search, що означає найкращий баланс між precision і recall.

Random Forest + SMOTE + Grid Search показує збалансований компроміс між precision (0.757) та recall (0.888).

### 1000

### 1 Baseline Random Forest

In [4]:
X = TH_1000.drop(columns=['label'])
y = TH_1000['label']

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2, stratify=y,random_state=42)
rf = RandomForestClassifier(random_state=42).fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

#
accuracy_rf = accuracy_score(y_test, y_pred)
precision_rf = precision_score(y_test, y_pred)
recall_rf = recall_score(y_test, y_pred)
f1_rf = f1_score(y_test, y_pred)
roc_auc_rf = roc_auc_score(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

print('1.Baseline Random Forest')
print("Accuracy:", accuracy_rf)
print("Precision:", precision_rf)
print("Recall:", recall_rf)
print("F1-score:", f1_rf)
print("ROC-AUC:", roc_auc_rf)


print("Confusion Matrix:")
print(cm)

1.Baseline Random Forest
Accuracy: 0.9392789373814042
Precision: 0.7837837837837838
Recall: 0.7837837837837838
F1-score: 0.7837837837837838
ROC-AUC: 0.9710969247393089
Confusion Matrix:
[[1311   48]
 [  48  174]]


### 2.Random Forest with Stratified K-Fold

In [5]:
X = TH_1000.drop(columns=['label'])
y = TH_1000['label']

rf = RandomForestClassifier(random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}


cv_results = cross_validate( rf,X,y,cv=skf, scoring=scoring)

print('2.Random Forest with Stratified K-Fold')
print("Accuracy:", cv_results['test_accuracy'].mean())
print("Precision:", cv_results['test_precision'].mean())
print("Recall:", cv_results['test_recall'].mean())
print("F1-score:", cv_results['test_f1'].mean())
print("ROC-AUC:", cv_results['test_roc_auc'].mean())

2.Random Forest with Stratified K-Fold
Accuracy: 0.9448450347881087
Precision: 0.8063356720428436
Recall: 0.8027027027027026
F1-score: 0.8038272823037979
ROC-AUC: 0.9784880907397463


### 3.Random Forest with Grid Search

In [6]:
X = TH_1000.drop(columns=['label'])
y = TH_1000['label']

rf = RandomForestClassifier(random_state=42)

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}


grid_search = GridSearchCV( estimator=rf,param_grid=param_grid,cv=5,scoring='f1',n_jobs=-1).fit(X_train, y_train)

best_rf = grid_search.best_estimator_

y_pred = best_rf.predict(X_test)
y_prob = best_rf.predict_proba(X_test)[:,1]

print('3.Random Forest with Grid Search')
print("Best parameters:", grid_search.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
3.Random Forest with Grid Search
Best parameters: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}
Accuracy: 0.9411764705882353
Precision: 0.7972350230414746
Recall: 0.7792792792792793
F1-score: 0.7881548974943052
ROC-AUC: 0.975051210150548print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

3.Random Forest with Grid Search
Best parameters: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}
Accuracy: 0.9411764705882353
Precision: 0.7972350230414746
Recall: 0.7792792792792793
F1-score: 0.7881548974943052
ROC-AUC: 0.975051210150548


### 4.Balanced Random Forest

In [7]:
brf = BalancedRandomForestClassifier(n_estimators=200,random_state=42).fit(X_train, y_train)

y_pred = brf.predict(X_test)
y_prob = brf.predict_proba(X_test)[:,1]

accuracy_brf = accuracy_score(y_test, y_pred)
precision_brf = precision_score(y_test, y_pred)
recall_brf = recall_score(y_test, y_pred)
f1_brf = f1_score(y_test, y_pred)
roc_auc_brf = roc_auc_score(y_test, y_prob)

print("4. Balanced Random Forest")
print("Accuracy:", accuracy_brf)
print("Precision:", precision_brf)
print("Recall:", recall_brf)
print("F1-score:", f1_brf)
print("ROC-AUC:", roc_auc_brf)


cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

4. Balanced Random Forest
Accuracy: 0.9234661606578115
Precision: 0.6711864406779661
Recall: 0.8918918918918919
F1-score: 0.7659574468085106
ROC-AUC: 0.9729663438272709
Confusion Matrix:
[[1262   97]
 [  24  198]]


### 5. Random Forest with SMOTE and Grid Search

In [10]:
pipeline = Pipeline([('smote', SMOTE(random_state=42)),('rf', RandomForestClassifier(random_state=42))])

param_grid = {
    'rf__n_estimators': [100, 200],
    'rf__max_depth': [None, 10, 20],
    'rf__min_samples_split': [2, 5],
    'rf__min_samples_leaf': [1, 2],
    'rf__max_features': ['sqrt', 'log2']
}

grid_search_smote = GridSearchCV(pipeline,param_grid,cv=5, scoring='f1',n_jobs=-1).fit(X_train, y_train)
best_smote_rf = grid_search_smote.best_estimator_

y_pred = best_smote_rf.predict(X_test)
y_prob = best_smote_rf.predict_proba(X_test)[:,1]

print("5. Random Forest with SMOTE and Grid Search")
print("Best parameters:", grid_search_smote.best_params_)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

5. Random Forest with SMOTE and Grid Search
Best parameters: {'rf__max_depth': None, 'rf__max_features': 'sqrt', 'rf__min_samples_leaf': 1, 'rf__min_samples_split': 5, 'rf__n_estimators': 100}
Accuracy: 0.9342188488298545
Precision: 0.736
Recall: 0.8288288288288288
F1-score: 0.7796610169491526
ROC-AUC: 0.9712411086583272
Confusion Matrix:
[[1293   66]
 [  38  184]]


Baseline Random Forest

Базова модель Random Forest продемонструвала високу загальну точність, де accuracy становить 0.939. Показники precision та recall є однаковими (0.784), що свідчить про збалансовану роботу моделі щодо виявлення подій buzz. Значення F1-score також дорівнює 0.784, що підтверджує рівновагу між точністю та повнотою. Значення ROC-AUC = 0.971 вказує на дуже хорошу здатність моделі розрізняти класи.

Аналіз матриці помилок показує, що модель припустилася однакової кількості хибно позитивних і хибно негативних прогнозів (по 48), що є ознакою стабільної поведінки моделі без перекосу в один із класів.

Random Forest з Stratified K-Fold Cross-Validation

Застосування стратифікованої крос-валідації дозволило отримати більш надійну оцінку якості моделі. У цьому випадку accuracy зросла до 0.945, що є найвищим значенням серед усіх моделей. Також покращилися показники precision (0.806) та recall (0.803), що призвело до збільшення F1-score до 0.804.

Значення ROC-AUC = 0.978 є найвищим серед усіх експериментів, що свідчить про найкращу здатність моделі відокремлювати класи. Це підтверджує, що використання крос-валідації дозволяє отримати більш стабільну і надійну модель.

Random Forest з Grid Search

Оптимізація гіперпараметрів за допомогою Grid Search не призвела до суттєвого покращення результатів. Accuracy становить 0.941, що близьке до baseline. Показники precision (0.797) та recall (0.779) є трохи нижчими, ніж у моделі з крос-валідацією. F1-score дорівнює 0.788, що також не перевищує результати попередніх моделей.

Проте значення ROC-AUC = 0.975 залишається високим, що підтверджує хорошу якість моделі. Загалом, у цьому випадку tuning гіперпараметрів не дав суттєвої переваги.

Balanced Random Forest

Модель Balanced Random Forest продемонструвала найкращі результати за метрикою recall (0.892), що означає значне покращення здатності моделі виявляти події buzz. Це підтверджується і зменшенням кількості хибно негативних прогнозів до 24.

Однак це покращення досягнуто за рахунок зниження precision до 0.671, що означає збільшення кількості хибно позитивних передбачень. Відповідно, accuracy знизилася до 0.923, а F1-score становить 0.766, що нижче за результати базової моделі.

Таким чином, Balanced Random Forest ефективно знаходить рідкісні події, але ціною зниження загальної точності.

Random Forest з SMOTE та Grid Search

Використання SMOTE у поєднанні з Grid Search дозволило досягти компромісного результату між precision та recall. Значення recall становить 0.829, що значно вище, ніж у базової моделі, але нижче, ніж у Balanced Random Forest. При цьому precision дорівнює 0.736, що є кращим результатом, ніж у Balanced Random Forest.

F1-score становить 0.780, що близьке до baseline, але з покращеним балансом між метриками. Accuracy дорівнює 0.934, що трохи нижче за baseline, але все ще залишається високим. Значення ROC-AUC = 0.971 підтверджує стабільну якість моделі.

Загальний висновок

Результати експерименту показують, що при збільшенні порогу σ до 1000 задача класифікації стає складнішою через зростання незбалансованості класів. У таких умовах різні підходи демонструють різні переваги.

Модель Random Forest з Stratified K-Fold Cross-Validation показала найкращі загальні результати за метриками accuracy, F1-score та ROC-AUC, що свідчить про її стабільність та узагальнювальну здатність.

Balanced Random Forest виявилася найефективнішою для виявлення подій buzz, досягнувши найвищого recall, однак це супроводжується зниженням precision і загальної точності.

Модель Random Forest з використанням SMOTE та Grid Search забезпечує найбільш збалансований компроміс між precision та recall, що робить її доцільною у випадках, коли важливо як виявляти події buzz, так і уникати хибних спрацьовувань.

Отже, вибір оптимальної моделі залежить від поставленої задачі: якщо пріоритетом є максимальне виявлення подій, доцільно використовувати Balanced Random Forest, тоді як для досягнення балансу між метриками — Random Forest із SMOTE.

### 1500

### 1 Baseline Random Forest

In [11]:
X = TH_1500.drop(columns=['label'])
y = TH_1500['label']

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2, stratify=y,random_state=42)
rf = RandomForestClassifier(random_state=42).fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

#
accuracy_rf = accuracy_score(y_test, y_pred)
precision_rf = precision_score(y_test, y_pred)
recall_rf = recall_score(y_test, y_pred)
f1_rf = f1_score(y_test, y_pred)
roc_auc_rf = roc_auc_score(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

print('1.Baseline Random Forest')
print("Accuracy:", accuracy_rf)
print("Precision:", precision_rf)
print("Recall:", recall_rf)
print("F1-score:", f1_rf)
print("ROC-AUC:", roc_auc_rf)


print("Confusion Matrix:")
print(cm)

1.Baseline Random Forest
Accuracy: 0.9626818469323213
Precision: 0.8562091503267973
Recall: 0.7797619047619048
F1-score: 0.8161993769470405
ROC-AUC: 0.9869346712499579
Confusion Matrix:
[[1391   22]
 [  37  131]]


### 2.Random Forest with Stratified K-Fold

In [12]:
X = TH_1500.drop(columns=['label'])
y = TH_1500['label']

rf = RandomForestClassifier(random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}


cv_results = cross_validate( rf,X,y,cv=skf, scoring=scoring)

print('2.Random Forest with Stratified K-Fold')
print("Accuracy:", cv_results['test_accuracy'].mean())
print("Precision:", cv_results['test_precision'].mean())
print("Recall:", cv_results['test_recall'].mean())
print("F1-score:", cv_results['test_f1'].mean())
print("ROC-AUC:", cv_results['test_roc_auc'].mean())

2.Random Forest with Stratified K-Fold
Accuracy: 0.9606578115117015
Precision: 0.8182158046451601
Recall: 0.809756269371654
F1-score: 0.8138728374058118
ROC-AUC: 0.9839505586701179


### 3.Random Forest with Grid Search

In [13]:
X = TH_1500.drop(columns=['label'])
y = TH_1500['label']

rf = RandomForestClassifier(random_state=42)

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}


grid_search = GridSearchCV( estimator=rf,param_grid=param_grid,cv=5,scoring='f1',n_jobs=-1).fit(X_train, y_train)

best_rf = grid_search.best_estimator_

y_pred = best_rf.predict(X_test)
y_prob = best_rf.predict_proba(X_test)[:,1]

print('3.Random Forest with Grid Search')
print("Best parameters:", grid_search.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

3.Random Forest with Grid Search
Best parameters: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}
Accuracy: 0.9614168247944339
Precision: 0.8543046357615894
Recall: 0.7678571428571429
F1-score: 0.8087774294670846
ROC-AUC: 0.9873875240117278


### 4.Balanced Random Forest

In [14]:
brf = BalancedRandomForestClassifier(n_estimators=200,random_state=42).fit(X_train, y_train)

y_pred = brf.predict(X_test)
y_prob = brf.predict_proba(X_test)[:,1]

accuracy_brf = accuracy_score(y_test, y_pred)
precision_brf = precision_score(y_test, y_pred)
recall_brf = recall_score(y_test, y_pred)
f1_brf = f1_score(y_test, y_pred)
roc_auc_brf = roc_auc_score(y_test, y_prob)

print("4. Balanced Random Forest")
print("Accuracy:", accuracy_brf)
print("Precision:", precision_brf)
print("Recall:", recall_brf)
print("F1-score:", f1_brf)
print("ROC-AUC:", roc_auc_brf)


cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

4. Balanced Random Forest
Accuracy: 0.946236559139785
Precision: 0.6796536796536796
Recall: 0.9345238095238095
F1-score: 0.7869674185463659
ROC-AUC: 0.9861427054898393
Confusion Matrix:
[[1339   74]
 [  11  157]]


### 5. Random Forest with SMOTE and Grid Search

In [15]:
pipeline = Pipeline([('smote', SMOTE(random_state=42)),('rf', RandomForestClassifier(random_state=42))])

param_grid = {
    'rf__n_estimators': [100, 200],
    'rf__max_depth': [None, 10, 20],
    'rf__min_samples_split': [2, 5],
    'rf__min_samples_leaf': [1, 2],
    'rf__max_features': ['sqrt', 'log2']
}

grid_search_smote = GridSearchCV(pipeline,param_grid,cv=5, scoring='f1',n_jobs=-1).fit(X_train, y_train)
best_smote_rf = grid_search_smote.best_estimator_

y_pred = best_smote_rf.predict(X_test)
y_prob = best_smote_rf.predict_proba(X_test)[:,1]

print("5. Random Forest with SMOTE and Grid Search")
print("Best parameters:", grid_search_smote.best_params_)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

5. Random Forest with SMOTE and Grid Search
Best parameters: {'rf__max_depth': None, 'rf__max_features': 'sqrt', 'rf__min_samples_leaf': 1, 'rf__min_samples_split': 2, 'rf__n_estimators': 100}
Accuracy: 0.956989247311828
Precision: 0.7604166666666666
Recall: 0.8690476190476191
F1-score: 0.8111111111111111
ROC-AUC: 0.9859636706770465
Confusion Matrix:
[[1367   46]
 [  22  146]]


In [16]:
f1_score(y_test, y_pred, pos_label=1)

0.8111111111111111

Baseline Random Forest

Базова модель Random Forest показала найвищу загальну точність серед усіх експериментів: accuracy = 0.963. Також було отримано високе значення precision = 0.856, що означає, що більшість передбачених подій buzz є правильними. Однак recall становить 0.780, що свідчить про те, що частина реальних подій buzz залишається невиявленою.

Значення F1-score = 0.816 підтверджує хороший баланс між precision та recall, а ROC-AUC = 0.987 вказує на відмінну здатність моделі розрізняти класи. Аналіз матриці помилок показує, що модель допустила лише 22 хибно позитивних та 37 хибно негативних прогнозів, що є дуже хорошим результатом з огляду на складність задачі.

Random Forest з Stratified K-Fold Cross-Validation

Модель із використанням стратифікованої крос-валідації продемонструвала дещо нижчу accuracy = 0.961, проте показала більш збалансовані результати. Значення precision = 0.818 та recall = 0.810 є близькими між собою, що призвело до F1-score = 0.814.

Значення ROC-AUC = 0.984 підтверджує стабільну якість моделі. У цьому випадку крос-валідація дозволяє отримати більш узагальнену оцінку, хоча й без суттєвого покращення показників.

Random Forest з Grid Search

Після оптимізації гіперпараметрів модель продемонструвала accuracy = 0.961, що практично відповідає результатам попередніх моделей. Значення precision = 0.854 залишилося високим, однак recall знизився до 0.768, що свідчить про дещо гіршу здатність виявляти події buzz.

F1-score становить 0.809, що трохи нижче за baseline. Водночас ROC-AUC = 0.987 є найвищим серед усіх моделей, що вказує на дуже хорошу здатність моделі до розділення класів. Таким чином, оптимізація гіперпараметрів не призвела до суттєвого покращення ключових метрик.

Balanced Random Forest

Balanced Random Forest знову показала найкращі результати за метрикою recall = 0.935, що означає майже повне виявлення подій buzz. Кількість хибно негативних прогнозів зменшилася до 11, що є найкращим результатом серед усіх моделей.

Однак це досягнуто за рахунок значного зниження precision до 0.680, що вказує на збільшення кількості хибно позитивних передбачень. Відповідно, accuracy знизилася до 0.946, а F1-score становить 0.787.

Значення ROC-AUC = 0.986 залишається дуже високим, що підтверджує загальну ефективність моделі, незважаючи на зниження precision.

Random Forest з SMOTE та Grid Search

Модель із використанням SMOTE у поєднанні з Grid Search демонструє збалансований компроміс між точністю та повнотою. Recall становить 0.869, що значно вище за baseline модель, але нижче, ніж у Balanced Random Forest. Precision дорівнює 0.760, що є суттєво кращим показником, ніж у Balanced Random Forest.

F1-score становить 0.811, що близьке до результатів базової моделі. Загальна точність (accuracy = 0.957) залишається високою, а ROC-AUC = 0.986 підтверджує стабільну якість класифікації.

Аналіз матриці помилок показує зменшення хибно негативних прогнозів (22) порівняно з baseline, що свідчить про покращення виявлення подій buzz.

Загальний висновок

Результати для датасету TH_1500 демонструють, що зі збільшенням порогу σ задача класифікації стає ще більш складною через зростаючу незбалансованість класів.

Базова модель Random Forest демонструє найвищу загальну точність і високий рівень precision, однак не забезпечує максимального виявлення подій buzz. Модель із крос-валідацією забезпечує більш стабільні результати, але без суттєвого покращення.

Balanced Random Forest, як і в попередніх експериментах, досягає найвищого recall, що робить її найбільш ефективною для задач, де критично важливо виявити всі події buzz. Водночас це супроводжується значним зниженням precision.

Модель Random Forest із використанням SMOTE та Grid Search демонструє найкращий баланс між precision та recall, що дозволяє ефективно виявляти події buzz без значного зростання кількості хибних спрацьовувань.

Отже, для задачі з високим рівнем незбалансованості найбільш доцільним є використання підходів, що враховують баланс класів, зокрема SMOTE або Balanced Random Forest, залежно від пріоритетів дослідження.

### Result

У результаті проведених експериментів було встановлено, що ефективність моделей залежить від рівня незбалансованості датасету (значення σ).

Для датасету TH_500 найкращий баланс між метриками продемонструвала модель Random Forest з Grid Search, яка досягла високого значення F1-score (0.822) та ROC-AUC (0.970), забезпечуючи стабільне поєднання precision і recall.

Для датасету TH_1000 найкращі результати показала модель Random Forest з Stratified K-Fold Cross-Validation, яка досягла найвищих значень accuracy (0.945), F1-score (0.804) та ROC-AUC (0.978), що свідчить про її стабільність і здатність узагальнювати результати.

Для найбільш незбалансованого датасету TH_1500 оптимальним вибором стала модель Random Forest з використанням SMOTE та Grid Search, яка забезпечила найкращий компроміс між precision (0.760) та recall (0.869), що особливо важливо при рідкісних подіях buzz.

Таким чином, зі збільшенням порогового значення σ та зростанням незбалансованості даних, ефективність стандартних моделей зменшується, і більш доцільним стає використання методів балансування класів, таких як SMOTE або Balanced Random Forest.

In [1]:
import pandas as pd

data_500 = [
    ["Baseline RF", 0.9209, 0.8141, 0.8305, 0.8222, 0.9683],
    ["RF Stratified K-Fold", 0.9164, 0.8103, 0.8110, 0.8104, 0.9637],
    ["RF Grid Search", 0.9203, 0.8083, 0.8362, 0.8220, 0.9701],
    ["Balanced RF", 0.9013, 0.7115, 0.9282, 0.8055, 0.9692],
    ["RF + SMOTE + Grid", 0.9127, 0.7574, 0.8879, 0.8175, 0.9671],
]

columns = ["Model", "Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]

df_500 = pd.DataFrame(data_500, columns=columns)
df_500

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Baseline RF,0.9209,0.8141,0.8305,0.8222,0.9683
1,RF Stratified K-Fold,0.9164,0.8103,0.8110,0.8104,0.9637
2,RF Grid Search,0.9203,0.8083,0.8362,0.8220,0.9701
3,Balanced RF,0.9013,0.7115,0.9282,0.8055,0.9692
4,RF + SMOTE + Grid,0.9127,0.7574,0.8879,0.8175,0.9671


In [2]:
data_1000 = [
    ["Baseline RF", 0.9393, 0.7838, 0.7838, 0.7838, 0.9711],
    ["RF Stratified K-Fold", 0.9448, 0.8063, 0.8027, 0.8038, 0.9785],
    ["RF Grid Search", 0.9412, 0.7972, 0.7793, 0.7882, 0.9751],
    ["Balanced RF", 0.9235, 0.6712, 0.8919, 0.7660, 0.9730],
    ["RF + SMOTE + Grid", 0.9342, 0.7360, 0.8288, 0.7797, 0.9712],
]

df_1000 = pd.DataFrame(data_1000, columns=columns)
df_1000

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Baseline RF,0.9393,0.7838,0.7838,0.7838,0.9711
1,RF Stratified K-Fold,0.9448,0.8063,0.8027,0.8038,0.9785
2,RF Grid Search,0.9412,0.7972,0.7793,0.7882,0.9751
3,Balanced RF,0.9235,0.6712,0.8919,0.7660,0.9730
4,RF + SMOTE + Grid,0.9342,0.7360,0.8288,0.7797,0.9712


In [3]:
data_1500 = [
    ["Baseline RF", 0.9627, 0.8562, 0.7798, 0.8162, 0.9869],
    ["RF Stratified K-Fold", 0.9607, 0.8182, 0.8098, 0.8139, 0.9840],
    ["RF Grid Search", 0.9614, 0.8543, 0.7679, 0.8088, 0.9874],
    ["Balanced RF", 0.9462, 0.6797, 0.9345, 0.7870, 0.9861],
    ["RF + SMOTE + Grid", 0.9570, 0.7604, 0.8690, 0.8111, 0.9860],
]

df_1500 = pd.DataFrame(data_1500, columns=columns)
df_1500

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Baseline RF,0.9627,0.8562,0.7798,0.8162,0.9869
1,RF Stratified K-Fold,0.9607,0.8182,0.8098,0.8139,0.9840
2,RF Grid Search,0.9614,0.8543,0.7679,0.8088,0.9874
3,Balanced RF,0.9462,0.6797,0.9345,0.7870,0.9861
4,RF + SMOTE + Grid,0.9570,0.7604,0.8690,0.8111,0.9860
